# GASSP1 direct coefficient-domain exact-recovery gate

## One question

**With the real GASSP1 trajectory and coil maps, but exact noiseless data and no forward-model mismatch, can a direct rank-5 coefficient model recover sharp anatomy from random initialization?**

The physical model is fixed in the forward path:

```text
signed IR dictionary -> per-curve L2 normalization -> SVD -> Phi
X(t,x,y) = sum_k Phi(t,k) C(k,x,y)
y = A(X)
```

Two arms isolate the failure source:

| Arm | Unknowns | Optimizer | Role |
|---|---|---|---|
| **A: `coeff_cg`** | five free complex coefficient maps | linear conjugate gradient on `A*Phi` | encoding/conditioning control |
| **B: `coeff_inr`** | 2D HashGrid -> five complex coefficient maps | Adam from random initialization | proposed direct coefficient INR |

Both arms use the same rank-5 basis, sharp synthetic truth, real trajectory/SMap, support, and uniform complex data-consistency definition. No measured k-space values, CG image, fully sampled image, temporal TV, spatial regularizer, DCF weighting, B0, noise, or real-data seed is used.

Interpretation:

- `coeff_cg` not converged -> extend/fix the linear solve before interpreting image error.
- `coeff_cg` converged but both arms fail -> `A*Phi` does not identify this sharp target without an added spatial prior.
- `coeff_cg` passes and `coeff_inr` fails -> INR capacity/optimization is the blocker.
- `coeff_inr` passes -> promote the direct coefficient model to the retrospective fully sampled-reference gate.


## 1. Stage only the fixed operator code and acquisition metadata

`gassp1_data.mat` is copied because it contains the trajectory, SMap, support, DCF, grid, and shift. Its measured `k_data` is deliberately removed from the in-memory dictionary before either arm is constructed.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
from datetime import datetime
import hashlib, json, os, shutil, sys, time, zipfile

EXPERIMENT_ID = datetime.now().strftime('%Y%m%d_%H%M%S_%f')
DRIVE_ROOT = Path('/content/drive/MyDrive/T1_blood_INR_v2')
DRIVE_DATA = DRIVE_ROOT / '02_data_reference'
DRIVE_CODE = DRIVE_ROOT / '03_code'
DRIVE_RESULTS = DRIVE_ROOT / 'results' / 'coeff_subspace_exact_gate' / EXPERIMENT_ID
LOCAL = Path('/content') / f'T1_blood_INR_coeff_exact_{EXPERIMENT_ID}'
CODE_ZIP = DRIVE_CODE / 'T1_blood_INR_code_subspace_validation.zip'
RAW_PATH = DRIVE_DATA / 'gassp1_data.mat'
if not RAW_PATH.exists():
    RAW_PATH = DRIVE_ROOT / 'gassp1_data.mat'

missing = [str(path) for path in [CODE_ZIP, RAW_PATH] if not path.exists()]
assert not missing, 'Missing Drive input:\n' + '\n'.join(missing)
LOCAL.mkdir(parents=True)
DRIVE_RESULTS.mkdir(parents=True)
with zipfile.ZipFile(CODE_ZIP) as archive:
    archive.extractall(LOCAL)
shutil.copy2(RAW_PATH, LOCAL / 'gassp1_data.mat')
os.chdir(LOCAL)

forbidden = ['cg_predeblur.mat', 'full_spiral_reference.mat', 'full_spiral_shared_curve.mat']
assert not any((LOCAL / name).exists() for name in forbidden)
print('work:', LOCAL)
print('results:', DRIVE_RESULTS)


## 2. Install and verify the immutable operator inputs


In [ ]:
!nvidia-smi
!pip -q install torchkbnufft ninja imageio pandas scipy matplotlib pillow h5py tqdm
!pip -q install git+https://github.com/NVlabs/tiny-cuda-nn/#subdirectory=bindings/torch

import h5py
import imageio.v2 as imageio
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import tinycudann as tcnn
from PIL import Image, ImageDraw
from IPython.display import display
from scipy import io
from scipy.ndimage import gaussian_filter
from tqdm.auto import tqdm

assert torch.cuda.is_available(), 'A CUDA GPU is required'
device = torch.device('cuda:0')

def sha256(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

required = ['load_spiral.py', 'spiral_nufft.py']
assert all(Path(name).exists() for name in required)
with h5py.File('gassp1_data.mat', 'r') as handle:
    assert handle['k_data'].shape == (28, 50, 3315)
    assert handle['Traj'].shape == (50, 3315)
print('operator inputs verified')


## 3. Build the protocol-specific rank-5 T1 basis

This uses the same mathematical construction as MATLAB `getT1Prior`: signed IR dictionary, column-wise L2 normalization, then the leading left singular vectors. The Gram matrix is accumulated in chunks only to avoid materializing the full 50 x 1,500,000 dictionary; its eigenvectors are the same left singular subspace as the SVD, up to basis-vector signs.


In [ ]:
TI_MS = 63 + 200 * np.arange(50, dtype=np.float64)
T1_GRID_MS = np.linspace(200, 5000, 1500, dtype=np.float64)
MZ0_OVER_M0 = np.linspace(-1.0, 0.0, 1000, dtype=np.float64)
RANK = 5

def build_ir_basis(ti_ms, t1_grid_ms, mz_grid, rank, t1_chunk=25):
    gram = np.zeros((ti_ms.size, ti_ms.size), dtype=np.float64)
    for start in range(0, t1_grid_ms.size, t1_chunk):
        t1 = t1_grid_ms[start:start + t1_chunk]
        curves = 1.0 + (mz_grid[None, None, :] - 1.0) * np.exp(
            -ti_ms[:, None, None] / t1[None, :, None])
        curves = curves.reshape(ti_ms.size, -1)
        curves /= np.linalg.norm(curves, axis=0, keepdims=True)
        gram += curves @ curves.T
    eigenvalues, eigenvectors = np.linalg.eigh(gram)
    order = np.argsort(eigenvalues)[::-1]
    eigenvalues = np.clip(eigenvalues[order], 0, None)
    basis = eigenvectors[:, order[:rank]]
    for column in range(rank):
        pivot = np.argmax(np.abs(basis[:, column]))
        if basis[pivot, column] < 0:
            basis[:, column] *= -1
    retained = float(eigenvalues[:rank].sum() / eigenvalues.sum())
    return basis.astype(np.float32), eigenvalues, retained

basis, dictionary_eigenvalues, retained_energy = build_ir_basis(
    TI_MS, T1_GRID_MS, MZ0_OVER_M0, RANK)
assert np.allclose(basis.T @ basis, np.eye(RANK), atol=1e-5)
assert retained_energy > 0.99999
BASIS_PATH = LOCAL / 'gassp1_ir_rank5_ti63_mzminus1to0.npy'
np.save(BASIS_PATH, basis)

basis_manifest = {
    'construction': 'signed IR curves -> per-curve L2 normalization -> left singular subspace',
    'implementation': 'chunked D*D.T eigendecomposition; projector-equivalent to SVD',
    'TI_ms': TI_MS.tolist(),
    'T1_grid_ms': [200, 5000, 1500],
    'Mz0_over_M0': [-1.0, 0.0, 1000],
    'rank': RANK,
    'retained_dictionary_energy': retained_energy,
    'basis_sha256': sha256(BASIS_PATH),
}
(LOCAL / 'basis_manifest.json').write_text(json.dumps(basis_manifest, indent=2))
display(pd.Series(basis_manifest, name='basis contract'))

plt.figure(figsize=(8, 3))
for column in range(RANK):
    plt.plot(TI_MS, basis[:, column], label=f'B{column + 1}')
plt.xlabel('effective TI (ms)')
plt.title('Protocol-specific rank-5 temporal basis')
plt.grid(True)
plt.legend(ncol=5)
plt.tight_layout()
plt.savefig(DRIVE_RESULTS / 'rank5_basis.png', dpi=140)
plt.show()


## 4. Construct a sharp physical phantom and exact GASSP measurements

The phantom contains hard tissue and vessel boundaries, different T1/Mz/M0 regions, and a smooth complex phase. Its physical IR series is projected once into `Phi`, making the declared rank-5 model exact while preserving the spatial edges. The real GASSP1 trajectory, 28 coil maps, grid shift, and support are retained.


In [ ]:
from load_spiral import load_spiral_data
from spiral_nufft import SpiralNUFFT

spiral = load_spiral_data('gassp1_data.mat', apply_shift=False)
measured_kdata_shape = list(spiral['kdata'].shape)
del spiral['kdata']  # The first gate must not use measured signal values.
N = int(spiral['N'])
support = spiral['mask'].astype(bool)
support_t = torch.as_tensor(support, dtype=torch.float32, device=device)
basis_t = torch.as_tensor(basis, dtype=torch.complex64, device=device)

operator = SpiralNUFFT(
    torch.as_tensor(spiral['ktraj']),
    torch.as_tensor(spiral['smap']),
    torch.as_tensor(spiral['wi']),
    N, device, sinv=None, support_mask=None, dcf_norm='mean',
    kb_grid_size=324,
    n_shift=(N / 2 + spiral['shift'][1], N / 2 + spiral['shift'][0]),
)

axis = (np.arange(N, dtype=np.float32) + 0.5) / N * 2 - 1
yy, xx = np.meshgrid(axis, axis, indexing='ij')
body = ((xx / 0.72) ** 2 + (yy / 0.88) ** 2) <= 1
ijv_left = (xx + 0.23) ** 2 + (yy - 0.05) ** 2 <= 0.095 ** 2
ijv_right = (xx - 0.23) ** 2 + (yy - 0.05) ** 2 <= 0.095 ** 2
carotid_left = (xx + 0.15) ** 2 + (yy + 0.17) ** 2 <= 0.052 ** 2
carotid_right = (xx - 0.15) ** 2 + (yy + 0.17) ** 2 <= 0.052 ** 2
muscle = body & ~ijv_left & ~ijv_right & ~carotid_left & ~carotid_right

m0 = np.zeros((N, N), np.float32)
t1_map = np.full((N, N), 1000.0, np.float32)
mz0 = np.full((N, N), -0.45, np.float32)
m0[muscle] = 0.62
t1_map[muscle] = 1050
m0[ijv_left | ijv_right] = 1.00
t1_map[ijv_left | ijv_right] = 1880
mz0[ijv_left | ijv_right] = -0.92
m0[carotid_left | carotid_right] = 0.88
t1_map[carotid_left | carotid_right] = 1550
mz0[carotid_left | carotid_right] = -0.82
m0 *= support
phase = np.exp(1j * (0.55 * xx - 0.35 * yy + 0.20 * xx * yy)).astype(np.complex64)
physical = m0[None] * (1 + (mz0[None] - 1) * np.exp(
    -TI_MS[:, None, None] / t1_map[None])) * phase[None]
scale = np.quantile(np.abs(physical[:, support]), 0.995)
physical = (physical / scale).astype(np.complex64)
true_coeff_np = (basis.T @ physical.reshape(50, -1)).reshape(RANK, N, N).astype(np.complex64)
true_coeff = torch.as_tensor(true_coeff_np, device=device) * support_t

def render_coeff(coeff):
    coeff = coeff * support_t
    return torch.einsum('tk,khw->thw', basis_t, coeff).unsqueeze(1)

def encode_coeff(coeff):
    return operator.forward(render_coeff(coeff))

def adjoint_coeff(kspace):
    image = operator.adjoint(kspace, weighted=False).squeeze(1)
    return torch.einsum('tk,thw->khw', basis_t.conj(), image) * support_t

with torch.no_grad():
    true_image = render_coeff(true_coeff)
    synthetic_kdata = encode_coeff(true_coeff)
    probe_coeff = (torch.randn_like(true_coeff) + 1j * torch.randn_like(true_coeff)) * 1e-3
    left = torch.vdot(encode_coeff(probe_coeff).reshape(-1), synthetic_kdata.reshape(-1))
    right = torch.vdot(probe_coeff.reshape(-1), adjoint_coeff(synthetic_kdata).reshape(-1))
    adjoint_rel_error = float((left - right).abs() / torch.maximum(left.abs(), right.abs()).clamp_min(1e-12))
assert adjoint_rel_error < 2e-5, adjoint_rel_error
print('coefficient operator adjoint relative error:', adjoint_rel_error)
print('synthetic k-space:', tuple(synthetic_kdata.shape))

fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for column, frame in enumerate([0, 12, 25, 49]):
    axes[0, column].imshow(np.abs(physical[frame]), cmap='gray')
    axes[0, column].set_title(f'physical TI={TI_MS[frame]:.0f} ms')
    axes[1, column].imshow(true_image[frame, 0].abs().cpu(), cmap='gray')
    axes[1, column].set_title('exact rank-5 target')
for axis_plot in axes.ravel():
    axis_plot.axis('off')
plt.tight_layout()
plt.savefig(DRIVE_RESULTS / 'sharp_phantom_target.png', dpi=140)
plt.show()


## 5. Common metrics and predeclared gate

The outer-k metric uses actual normalized trajectory radius, not readout index. No scale or phase alignment is applied: the exact forward data determine both.


In [ ]:
radius = np.sqrt(np.sum(np.asarray(spiral['ktraj']) ** 2, axis=1))
radius /= radius.max()
SHELLS = [(0.00, 0.25), (0.25, 0.50), (0.50, 0.75), (0.75, 1.01)]

def relative_energy(prediction, target, sample_mask=None):
    error = (prediction - target).abs().square()
    power = target.abs().square()
    if sample_mask is not None:
        weight = torch.as_tensor(sample_mask, dtype=error.dtype, device=error.device)[:, None, :]
        error = error * weight
        power = power * weight
    return float(error.sum() / power.sum().clamp_min(1e-20))

def gradient_nrmse(prediction, target):
    pred = np.abs(prediction)
    truth = np.abs(target)
    error = sum(np.linalg.norm(np.diff(pred, axis=axis) - np.diff(truth, axis=axis)) ** 2
                for axis in [1, 2])
    power = sum(np.linalg.norm(np.diff(truth, axis=axis)) ** 2 for axis in [1, 2])
    return float(np.sqrt(error / power))

def highpass_nrmse(prediction, target):
    pred = np.abs(prediction)
    truth = np.abs(target)
    pred_high = pred - gaussian_filter(pred, sigma=(0, 1, 1))
    truth_high = truth - gaussian_filter(truth, sigma=(0, 1, 1))
    return float(np.linalg.norm(pred_high - truth_high) / np.linalg.norm(truth_high))

def score_arm(name, coeff, normal_residual=None):
    with torch.no_grad():
        image = render_coeff(coeff)
        kspace = encode_coeff(coeff)
    image_np = image[:, 0].cpu().numpy()
    target_np = true_image[:, 0].cpu().numpy()
    coeff_np = coeff.cpu().numpy()
    row = {
        'arm': name,
        'normal_residual': None if normal_residual is None else float(normal_residual),
        'uniform_DC': relative_energy(kspace, synthetic_kdata),
        'complex_NRMSE': float(np.linalg.norm(image_np - target_np) / np.linalg.norm(target_np)),
        'magnitude_NRMSE': float(np.linalg.norm(np.abs(image_np) - np.abs(target_np)) / np.linalg.norm(np.abs(target_np))),
        'gradient_NRMSE': gradient_nrmse(image_np, target_np),
        'highpass_NRMSE': highpass_nrmse(image_np, target_np),
        'coefficient_NRMSE': float(np.linalg.norm(coeff_np - true_coeff_np) / np.linalg.norm(true_coeff_np)),
    }
    for low, high in SHELLS:
        key = f'shell_{low:.2f}_{high:.2f}_DC'
        row[key] = relative_energy(kspace, synthetic_kdata, (radius >= low) & (radius < high))
    return row, image_np

def image_gate(row):
    return (row['uniform_DC'] < 0.01
            and row['magnitude_NRMSE'] < 0.05
            and row['gradient_NRMSE'] < 0.15
            and row['shell_0.75_1.01_DC'] < 0.05)

print('Gate: DC < 0.01, magnitude NRMSE < 0.05, gradient NRMSE < 0.15, outer-shell DC < 0.05')


## 6. Arm A — free coefficient maps solved by CG

This is not the proposed final reconstruction. It asks whether the linear coefficient system can recover the exact target without the INR parameterization. Image failure is interpreted only after the normal-equation residual has converged.


In [ ]:
CG_ITERS = 80
CG_TOL = 1e-6

def normal_coeff(coeff):
    return adjoint_coeff(encode_coeff(coeff))

@torch.no_grad()
def conjugate_gradient(rhs, iterations, tolerance):
    solution = torch.zeros_like(rhs)
    residual = rhs - normal_coeff(solution)
    direction = residual.clone()
    residual_power = torch.vdot(residual.reshape(-1), residual.reshape(-1)).real
    initial_power = residual_power.clone()
    history = []
    for iteration in tqdm(range(iterations), desc='coeff-CG'):
        normal_direction = normal_coeff(direction)
        denominator = torch.vdot(direction.reshape(-1), normal_direction.reshape(-1)).real
        if denominator <= 0:
            raise RuntimeError(f'non-positive CG curvature at iteration {iteration + 1}')
        step = residual_power / denominator
        solution += step * direction
        residual -= step * normal_direction
        new_power = torch.vdot(residual.reshape(-1), residual.reshape(-1)).real
        relative = float(torch.sqrt(new_power / initial_power).cpu())
        history.append({'iteration': iteration + 1, 'normal_residual': relative})
        if relative < tolerance:
            break
        direction = residual + (new_power / residual_power) * direction
        residual_power = new_power
    return solution * support_t, pd.DataFrame(history)

with torch.no_grad():
    cg_rhs = adjoint_coeff(synthetic_kdata)
    coeff_cg, cg_history = conjugate_gradient(cg_rhs, CG_ITERS, CG_TOL)
cg_normal_residual = float(cg_history.normal_residual.iloc[-1])
cg_score, cg_image = score_arm('coeff_cg', coeff_cg, cg_normal_residual)
display(pd.Series(cg_score))
cg_history.to_csv(DRIVE_RESULTS / 'coeff_cg_history.csv', index=False)
if cg_normal_residual >= 1e-3:
    print('CG IS NOT CONVERGED: extend/fix the solve before using its image error to judge A*Phi.')


## 7. Arm B — direct coefficient INR

A 2D HashGrid sees only `(y,x)` and emits `2*K` real channels interpreted as K complex coefficient maps. All hash levels are active: this first gate tests representation and optimization capacity, not a coarse-to-fine schedule. The only loss is uniform global complex relative L2 in synthetic k-space.


In [ ]:
SEED = 0
INR_EPOCHS = 800
INR_LR = 1e-3
RUN_INR = True
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
np.random.seed(SEED)

class CoefficientINR(torch.nn.Module):
    def __init__(self, grid_size, rank):
        super().__init__()
        levels = 16
        base_resolution = 16
        self.grid_size = grid_size
        self.rank = rank
        self.encoding = tcnn.Encoding(
            n_input_dims=2,
            encoding_config={
                'otype': 'HashGrid',
                'n_levels': levels,
                'n_features_per_level': 2,
                'log2_hashmap_size': 20,
                'base_resolution': base_resolution,
                'per_level_scale': float((grid_size / base_resolution) ** (1 / (levels - 1))),
            },
        )
        self.network = tcnn.Network(
            n_input_dims=self.encoding.n_output_dims,
            n_output_dims=2 * rank,
            network_config={
                'otype': 'FullyFusedMLP',
                'activation': 'ReLU',
                'output_activation': 'None',
                'n_neurons': 128,
                'n_hidden_layers': 3,
            },
        )

    def forward(self, coordinates):
        values = self.network(self.encoding(coordinates)).float()
        values = values.reshape(self.grid_size, self.grid_size, self.rank, 2)
        return torch.view_as_complex(values.contiguous()).permute(2, 0, 1)

coordinate_axis = torch.linspace(1 / (2 * N), 1 - 1 / (2 * N), N, device=device)
coord_y, coord_x = torch.meshgrid(coordinate_axis, coordinate_axis, indexing='ij')
coordinates = torch.stack([coord_y.reshape(-1), coord_x.reshape(-1)], dim=1)
coeff_model = CoefficientINR(N, RANK).to(device)
optimizer = torch.optim.Adam(coeff_model.parameters(), lr=INR_LR)
initial_model_path = LOCAL / 'coeff_inr_initial_seed0.pt'
torch.save(coeff_model.state_dict(), initial_model_path)
inr_history = []
target_power = synthetic_kdata.abs().square().sum().detach()

if RUN_INR:
    start_time = time.time()
    loop = tqdm(range(INR_EPOCHS), desc='coeff-INR')
    for epoch in loop:
        coeff = coeff_model(coordinates) * support_t
        prediction = encode_coeff(coeff)
        dc = (prediction - synthetic_kdata).abs().square().sum() / target_power
        optimizer.zero_grad(set_to_none=True)
        dc.backward()
        optimizer.step()
        inr_history.append({'epoch': epoch + 1, 'uniform_DC': float(dc.detach().cpu())})
        if (epoch + 1) % 20 == 0:
            loop.set_postfix(dc=f'{float(dc):.3e}')
    elapsed_seconds = time.time() - start_time
else:
    elapsed_seconds = 0.0

with torch.no_grad():
    coeff_inr = coeff_model(coordinates) * support_t
inr_score, inr_image = score_arm('coeff_inr', coeff_inr)
display(pd.Series(inr_score))
pd.DataFrame(inr_history).to_csv(DRIVE_RESULTS / 'coeff_inr_history.csv', index=False)
torch.save(coeff_model.state_dict(), DRIVE_RESULTS / 'coeff_inr_final_seed0.pt')
print(f'INR runtime: {elapsed_seconds / 60:.1f} min')


## 8. Joint result, fixed-window evidence, and decision

The GIF order is target | coefficient-CG | coefficient-INR. The notebook promotes only the direct coefficient formulation; it does not authorize real-data training. The next gate after a pass is retrospective undersampling of the fully sampled reference.


In [ ]:
scores = pd.DataFrame([cg_score, inr_score]).set_index('arm')
scores['image_gate_pass'] = [image_gate(cg_score), image_gate(inr_score)]
scores.to_csv(DRIVE_RESULTS / 'exact_gate_summary.csv')
display(scores)

CG_SOLVER_CONVERGED = cg_normal_residual < 1e-3
CG_IMAGE_PASS = bool(image_gate(cg_score))
INR_PASS = bool(image_gate(inr_score))
if not CG_SOLVER_CONVERGED:
    DECISION = 'STOP_CG_NOT_CONVERGED'
elif CG_IMAGE_PASS and not INR_PASS:
    DECISION = 'STOP_FIX_INR_PARAMETERIZATION_OR_OPTIMIZATION'
elif not CG_IMAGE_PASS and not INR_PASS:
    DECISION = 'STOP_A_PHI_NEEDS_SPATIAL_PRIOR_OR_BETTER_CONDITIONING'
elif INR_PASS and not CG_IMAGE_PASS:
    DECISION = 'PROMOTE_WITH_CAUTION_INR_PRIOR_IS_ESSENTIAL'
else:
    DECISION = 'PROMOTE_TO_FULL_REFERENCE_RETROSPECTIVE_GATE'
print('DECISION =', DECISION)

target_image_np = true_image[:, 0].cpu().numpy()
fixed_vmax = np.quantile(np.abs(target_image_np)[:, support], 0.995)

def frame_u8(image, frame):
    return (255 * np.clip(np.abs(image[frame]) / fixed_vmax, 0, 1)).astype(np.uint8)

gif_frames = []
for frame in range(50):
    canvas = Image.new('L', (3 * N, N + 24), color=0)
    draw = ImageDraw.Draw(canvas)
    for column, (label, image) in enumerate([
        ('target', target_image_np), ('coeff-CG', cg_image), ('coeff-INR', inr_image)]):
        canvas.paste(Image.fromarray(frame_u8(image, frame)), (column * N, 24))
        draw.text((column * N + 5, 5), label, fill=255)
    gif_frames.append(np.asarray(canvas.convert('RGB')))
GIF_PATH = DRIVE_RESULTS / 'target__coeff_cg__coeff_inr_fixed_window.gif'
imageio.mimsave(GIF_PATH, gif_frames, duration=0.18, loop=0)

fig, axes = plt.subplots(3, 4, figsize=(12, 9))
for row, (label, image) in enumerate([
        ('target', target_image_np), ('coeff-CG', cg_image), ('coeff-INR', inr_image)]):
    for column, frame in enumerate([0, 12, 25, 49]):
        axes[row, column].imshow(np.abs(image[frame]), cmap='gray', vmin=0, vmax=fixed_vmax)
        axes[row, column].set_title(f'{label}, TI={TI_MS[frame]:.0f} ms')
        axes[row, column].axis('off')
plt.tight_layout()
plt.savefig(DRIVE_RESULTS / 'selected_frames_fixed_window.png', dpi=140)
plt.show()

io.savemat(DRIVE_RESULTS / 'exact_gate_reconstructions.mat', {
    'basis': basis,
    'true_coeff': true_coeff_np,
    'coeff_cg': coeff_cg.cpu().numpy(),
    'coeff_inr': coeff_inr.cpu().numpy(),
    'target_image': target_image_np,
    'image_cg': cg_image,
    'image_inr': inr_image,
})


## 9. Provenance manifest


In [ ]:
manifest = {
    'experiment_id': EXPERIMENT_ID,
    'question': 'Can direct rank-5 coefficient reconstruction recover sharp exact synthetic anatomy with the real GASSP1 encoding?',
    'basis': basis_manifest,
    'phantom': {
        'construction': 'piecewise physical T1/Mz/M0 maps with hard vessel boundaries, then exact projection into Phi',
        'complex_phase': 'fixed smooth spatial phase',
    },
    'operator': {
        'trajectory_and_maps': 'gassp1_data.mat',
        'measured_kdata_shape_but_values_deleted': measured_kdata_shape,
        'apply_shift_ramp': False,
        'kb_grid_size': 324,
        'n_shift': [float(N / 2 + spiral['shift'][1]), float(N / 2 + spiral['shift'][0])],
        'B0': 'absent; exact matched synthetic gate',
        'coefficient_adjoint_relative_error': adjoint_rel_error,
    },
    'arms': {
        'coeff_cg': {'iterations': CG_ITERS, 'tolerance': CG_TOL, 'regularizer': 'none'},
        'coeff_inr': {
            'seed': SEED, 'epochs': INR_EPOCHS, 'lr': INR_LR,
            'input': '2D y,x', 'output': '5 complex coefficient maps',
            'hash_levels': 16, 'features_per_level': 2, 'log2_hashmap_size': 20,
            'MLP': '3x128 ReLU', 'regularizer': 'none',
        },
    },
    'data_loss': 'uniform global complex relative L2',
    'excluded': ['measured k-space values', 'CG image', 'fully sampled image', 'B0', 'noise', 'DCF weighting', 'temporal TV', 'spatial TV', 'LLR', 'wavelet'],
    'gate': {
        'uniform_DC': '<0.01', 'magnitude_NRMSE': '<0.05',
        'gradient_NRMSE': '<0.15', 'outer_shell_DC': '<0.05',
    },
    'scores': json.loads(scores.reset_index().to_json(orient='records')),
    'decision': DECISION,
    'code_sha256': {name: sha256(name) for name in required},
    'raw_container_sha256': sha256('gassp1_data.mat'),
}
shutil.copy2(BASIS_PATH, DRIVE_RESULTS / BASIS_PATH.name)
shutil.copy2(LOCAL / 'basis_manifest.json', DRIVE_RESULTS / 'basis_manifest.json')
shutil.copy2(initial_model_path, DRIVE_RESULTS / initial_model_path.name)
(DRIVE_RESULTS / 'experiment_manifest.json').write_text(json.dumps(manifest, indent=2))
print('evidence package:', DRIVE_RESULTS)
print('decision:', DECISION)
print('Next only after promotion: retrospective full-reference coefficient-CG vs coefficient-INR.')
